# Amazon Bedrock Converse API: Building Blocks & Weather Tool Calling

*This notebook explores the core building blocks of the **Amazon Bedrock Converse API** and walks through an end-to-end tool calling implementation with an external weather service using **Amazon first-party models** (Amazon Nova).*

---

## What is the Converse API?

The **Amazon Bedrock Converse API** provides a standardized, model-agnostic interface for conversational applications. Unlike raw foundation model APIs which require custom provider schemas (`inputText`, `messages-v1`, `prompt`), the Converse API standardizes:
- **Conversation State**: Structured message arrays (`user` and `assistant` turns).
- **System Steering**: First-class `system` prompts to govern persona and safety.
- **Inference Settings**: Universal `inferenceConfig` (`temperature`, `maxTokens`, `topP`).
- **Tool Use (Function Calling)**: Seamless tool specifications and structured `toolUse` / `toolResult` exchanges.

### Architecture: Weather Tool Execution Flow

<div align="center">
  <img src="../images/weather_tool_flow.svg" alt="Converse API Weather Tool Flow" width="100%" />
</div>

---

## 1. Environment Setup & Amazon Bedrock Client Initialization

We initialize the `bedrock-runtime` and `lambda` clients in our target region (`us-east-2`).

> [!IMPORTANT]
> **Amazon Nova Pro Model**:
> We use **Amazon Nova Pro** (`us.amazon.nova-pro-v1:0`), Amazon's flagship foundation model designed for complex reasoning, multi-step problem solving, and precise tool calling.
> Because the Bedrock Converse API is fully model-agnostic, the exact same `messages`, `system`, and `toolConfig` structures work seamlessly with Nova Pro using its System Inference Profile (`us.amazon.nova-pro-v1:0`).


In [8]:
import boto3
import json
import datetime
from botocore.config import Config
from botocore.exceptions import ClientError

# Target AWS Region
REGION = "us-east-2"

client_config = Config(
    region_name=REGION,
    connect_timeout=60,
    read_timeout=120,
    retries={"max_attempts": 3, "mode": "standard"}
)

# Initialize Bedrock Runtime & Lambda clients
bedrock = boto3.client(service_name="bedrock-runtime", config=client_config)
lambda_client = boto3.client(service_name="lambda", config=client_config)

# Amazon Nova models with required 'us.' System Inference Profile prefix
NOVA_PRO = "us.amazon.nova-pro-v1:0"
NOVA_LITE = "us.amazon.nova-lite-v1:0"
NOVA_MICRO = "us.amazon.nova-micro-v1:0"

# Target Model: Amazon Nova Pro
MODEL_ID = NOVA_PRO

print(f"Bedrock Runtime initialized for region: {REGION}")
print(f"Active Amazon Model: {MODEL_ID}")


Bedrock Runtime initialized for region: us-east-2
Active Amazon Model: us.amazon.nova-pro-v1:0


## 2. Building Block 1: System Prompts and Message Arrays

### A. System Prompts
A system prompt defines the model's behavior, personality, and operating constraints. It sets the foundation for how the model interacts and how strictly it adheres to formatting instructions.

### B. Message Arrays
Message arrays store conversation history as an ordered sequence of turns. Each turn contains:
- `role`: `'user'` or `'assistant'`
- `content`: A list of content blocks (e.g. `{'text': '...'}` or `{'image': ...}` or `{'toolUse': ...}` / `{'toolResult': ...}`).

Let's test the playlist assistant example from `plan.md`:

In [9]:
# 1. Define system prompt establishing persona and strict output constraints
system_prompt = [{
    "text": "You are an AI assistant that helps create playlists. Only return song names and artists."
}]

# 2. Message array containing initial user query
messages = [
    {
        "role": "user",
        "content": [{"text": "Create a playlist of 3 pop songs"}]
    }
]

print("=== User Query ===")
print(messages[0]["content"][0]["text"])

# 3. Invoke Converse API with Amazon Nova Pro
response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompt,
    inferenceConfig={"temperature": 0.7, "maxTokens": 250}
)

# 4. Extract assistant response
assistant_msg = response["output"]["message"]
print("\n=== Assistant Playlist Output ===")
print(assistant_msg["content"][0]["text"])

# 5. Inspect usage metrics
print("\nToken Usage:", response.get("usage"))


=== User Query ===
Create a playlist of 3 pop songs

=== Assistant Playlist Output ===
1. "Levitating" - Dua Lipa
2. "Dynamite" - BTS
3. "Blinding Lights" - The Weeknd

Token Usage: {'inputTokens': 25, 'outputTokens': 33, 'totalTokens': 58}


## 3. Building Block 2: Inference Parameters & Token Usage

Inference parameters control how the model generates tokens:
- **`temperature`** (0.0 to 1.0): Controls randomness. Lower values (0.0–0.2) yield focused, deterministic outputs; higher values (0.7–1.0) encourage creative variety.
- **`maxTokens`**: The maximum number of tokens to generate in the response.
- **`topP`** (0.0 to 1.0): Nucleus sampling threshold.
- **`additionalModelRequestFields`**: Provider-specific extensions (e.g., `topK` for models that support it).

Let's compare deterministic (`temperature=0.1`) vs creative (`temperature=0.9`) behavior:

In [10]:
def test_temperature(temp: float, prompt_text: str):
    resp = bedrock.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt_text}]}],
        system=[{"text": "You are a creative naming consultant. Generate 1 catchy product name with a 1-sentence tagline."}],
        inferenceConfig={"temperature": temp, "maxTokens": 100, "topP": 0.9}
    )
    text = resp["output"]["message"]["content"][0]["text"].strip()
    usage = resp.get("usage", {})
    print(f"[Temperature {temp}] Tokens: {usage.get('totalTokens')}")
    print(f"{text}\n")

sample_prompt = "Name an eco-friendly solar-powered smart water bottle."
print(f"Testing Prompt: '{sample_prompt}'\n")

print("--- Run A: Low Temperature (0.1 - Deterministic) ---")
test_temperature(0.1, sample_prompt)

print("--- Run B: High Temperature (0.9 - Creative) ---")
test_temperature(0.9, sample_prompt)


Testing Prompt: 'Name an eco-friendly solar-powered smart water bottle.'

--- Run A: Low Temperature (0.1 - Deterministic) ---
[Temperature 0.1] Tokens: 55
**SolarSip EcoSmart** – "Quench your thirst sustainably with every solar-powered sip."

--- Run B: High Temperature (0.9 - Creative) ---
[Temperature 0.9] Tokens: 60
**SolarSip Harmony** - Embrace the future of hydration with a bottle that sips sunshine and quenches your thirst sustainably.



## 4. Production Architecture: Context Window & Session Management

### Context Window Limits
The context window defines how much text (input + output tokens) the model can process at once:
- **Amazon Nova Pro**: 300,000 tokens (Flagship reasoning & tool calling capacity)
- **Amazon Nova Lite**: 300,000 tokens (Cost-efficient multimodal)
- **Amazon Nova Micro**: 128,000 tokens (Text-only, low-latency)
- **Amazon Titan Text Express**: 8,192 tokens

### Production Best Practices
1. **External State Storage**: In production, never keep conversation history in local process memory. Persist message histories in **Amazon DynamoDB** or **Amazon ElastiCache (Redis)** indexed by `session_id`.
2. **Sliding Window / Pruning**: When conversations grow long, prune older user/assistant turns or summarize earlier messages to avoid paying for redundant input tokens on every turn.
3. **Token Monitoring**: Always extract `response['usage']['totalTokens']` to monitor cost and prevent hitting context limits.


In [11]:
class ConversationSession:
    """
    Production helper demonstrating sliding-window memory management and token tracking.
    """
    def __init__(self, session_id: str, max_turns: int = 4):
        self.session_id = session_id
        self.max_turns = max_turns
        self.messages = []
        self.total_tokens_consumed = 0

    def add_user_message(self, text: str):
        self.messages.append({"role": "user", "content": [{"text": text}]})
        self._prune_history()

    def add_assistant_message(self, message: dict, usage: dict):
        self.messages.append(message)
        self.total_tokens_consumed += usage.get("totalTokens", 0)
        self._prune_history()

    def _prune_history(self):
        # Maintain the most recent N turns
        if len(self.messages) > (self.max_turns * 2):
            # Truncate oldest turns while ensuring first message starts with 'user'
            self.messages = self.messages[-(self.max_turns * 2):]

# Test session tracking
session = ConversationSession(session_id="demo-session-123", max_turns=2)
session.add_user_message("Hello, I need assistance with cloud architecture.")
print(f"Session initialized: {session.session_id} | Turns in memory: {len(session.messages)}")


Session initialized: demo-session-123 | Turns in memory: 1


## 5. Building Block 3: Tool Configuration (`toolConfig`)

Tool configurations allow the model to execute real-world functions by generating structured function arguments.

### Structure of `toolConfig`
- **`tools`**: An array of tool specifications.
  - `name`: Alphanumeric name of the tool (e.g. `getWeather`).
  - `description`: Explains what the tool does and what input values it expects.
  - `inputSchema`: A standard JSON Schema defining properties, types, and required fields.
- **`toolChoice`**:
  - `{"auto": {}}`: The model decides autonomously whether to call a tool or answer directly.
  - `{"any": {}}`: Forces the model to call at least one tool.
  - `{"tool": {"name": "getWeather"}}`: Forces the model to call the specific designated tool.

Let's define the **`weather_tool`** schema specified in `plan.md`:

In [12]:
# Define toolSpec for the weather tool
weather_tool = {
    "toolSpec": {
        "name": "getWeather",
        "description": "Gets the current weather using latitude and longitude.",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "lat": {
                        "type": "number",
                        "description": "The latitude coordinate of the location."
                    },
                    "lon": {
                        "type": "number",
                        "description": "The longitude coordinate of the location."
                    }
                },
                "required": ["lat", "lon"]
            }
        }
    }
}

# Assemble complete toolConfig
tool_config = {
    "tools": [weather_tool],
    "toolChoice": {"auto": {}}
}

print("Registered Tool Specification:")
print(json.dumps(tool_config, indent=2))


Registered Tool Specification:
{
  "tools": [
    {
      "toolSpec": {
        "name": "getWeather",
        "description": "Gets the current weather using latitude and longitude.",
        "inputSchema": {
          "json": {
            "type": "object",
            "properties": {
              "lat": {
                "type": "number",
                "description": "The latitude coordinate of the location."
              },
              "lon": {
                "type": "number",
                "description": "The longitude coordinate of the location."
              }
            },
            "required": [
              "lat",
              "lon"
            ]
          }
        }
      }
    }
  ],
  "toolChoice": {
    "auto": {}
  }
}


## 6. Weather Tool Implementation: AWS Lambda & Dual Dispatcher

In production, your application calls an **AWS Lambda function** (e.g. `weather-tool`).

To make this notebook **completely self-contained and immediately executable without requiring prior AWS Lambda deployment**, we implement a **Dual-Mode Dispatcher**:
1. Attempts to invoke live AWS Lambda function (`weather-tool`) if present in your account.
2. If the Lambda function is not deployed, it falls back seamlessly to a realistic weather provider mock (returning actual climate conditions for Portland, Seattle, New York, London, Tokyo, etc.).

In [13]:
# Known reference coordinates for realistic offline resolution
GEO_WEATHER_DB = [
    {"city": "Portland, OR", "lat": 45.5152, "lon": -122.6784, "temp": "62°F", "condition": "Partly Cloudy", "humidity": "68%"},
    {"city": "Portland, ME", "lat": 43.6591, "lon": -70.2568, "temp": "54°F", "condition": "Breezy", "humidity": "60%"},
    {"city": "Seattle, WA", "lat": 47.6062, "lon": -122.3321, "temp": "59°F", "condition": "Light Rain", "humidity": "78%"},
    {"city": "New York, NY", "lat": 40.7128, "lon": -74.0060, "temp": "68°F", "condition": "Sunny", "humidity": "52%"},
    {"city": "San Francisco, CA", "lat": 37.7749, "lon": -122.4194, "temp": "61°F", "condition": "Foggy", "humidity": "75%"},
    {"city": "London, UK", "lat": 51.5074, "lon": -0.1278, "temp": "15°C", "condition": "Overcast", "humidity": "70%"},
    {"city": "Tokyo, Japan", "lat": 35.6762, "lon": 139.6503, "temp": "20°C", "condition": "Clear", "humidity": "45%"}
]

def invoke_weather_lambda(input_data: dict, lambda_name: str = "weather-tool") -> dict:
    """
    Invokes the AWS Lambda function 'weather-tool'.
    If the function is not deployed in AWS, falls back seamlessly to the local geo-weather database.
    """
    # 1. Attempt AWS Lambda execution
    try:
        response = lambda_client.invoke(
            FunctionName=lambda_name,
            InvocationType="RequestResponse",
            Payload=json.dumps(input_data)
        )
        raw_payload = response["Payload"].read().decode("utf-8")
        result = json.loads(raw_payload)
        body = result.get("body", result)
        parsed_body = json.loads(body) if isinstance(body, str) else body
        print(f"[AWS Lambda '{lambda_name}'] Returned live payload: {parsed_body}")
        return parsed_body
    except (ClientError, Exception) as e:
        # 2. Seamless local fallback
        lat = float(input_data.get("lat", 0))
        lon = float(input_data.get("lon", 0))
        print(f"[Local Weather Dispatcher] Resolving coordinates (lat: {lat:.4f}, lon: {lon:.4f})...")
        
        # Find closest match from database
        def distance(item):
            return (item["lat"] - lat)**2 + (item["lon"] - lon)**2
        
        closest = min(GEO_WEATHER_DB, key=distance)
        local_result = {
            "location": closest["city"],
            "latitude": lat,
            "longitude": lon,
            "temperature": closest["temp"],
            "condition": closest["condition"],
            "humidity": closest["humidity"],
            "source": "Local Geo-Weather Service"
        }
        return local_result

# Test the dispatcher with Portland, OR coordinates (lat: 45.5152, lon: -122.6784)
test_output = invoke_weather_lambda({"lat": 45.5152, "lon": -122.6784})
print("Sample Dispatcher Output:", json.dumps(test_output, indent=2))


[Local Weather Dispatcher] Resolving coordinates (lat: 45.5152, lon: -122.6784)...
Sample Dispatcher Output: {
  "location": "Portland, OR",
  "latitude": 45.5152,
  "longitude": -122.6784,
  "temperature": "62\u00b0F",
  "condition": "Partly Cloudy",
  "humidity": "68%",
  "source": "Local Geo-Weather Service"
}


## 7. The End-to-End Tool Calling Loop

Now we orchestrate the complete tool execution cycle using **Amazon Nova Pro**:
1. **Initial Query**: User asks: *"What's the weather like in portland?"*
2. **System Prompt**: Instructs the model on role and disambiguation rules.
3. **Bedrock Detection**: Model determines it needs external live weather data and issues `stopReason='tool_use'` with `toolUseId` and coordinate arguments.
4. **Execution**: Client dispatches execution to `invoke_weather_lambda`.
5. **Tool Result Block**: Result is formatted into a message with `role: 'user'` and a `toolResult` content block.
6. **Final Synthesis**: Bedrock incorporates the tool output into a conversational response.


In [14]:
# 1. Initial user message
initial_user_msg = {
    "role": "user",
    "content": [{"text": "What's the weather like in portland?"}]
}

# 2. System prompt with persona and geographic disambiguation constraint
system_prompt = [{
    "text": "You are a helpful assistant that can check the weather using tools when needed. "
            "If the user inputs a city name that exists in multiple geographic areas, ask the user for clarification."
}]

print("=== Step 1: Sending Initial User Message to Amazon Nova ===")
print(f"User: {initial_user_msg['content'][0]['text']}\n")

# 3. Initial Converse call with toolConfig
initial_response = bedrock.converse(
    modelId=MODEL_ID,
    system=system_prompt,
    messages=[initial_user_msg],
    toolConfig=tool_config,
    inferenceConfig={"temperature": 0.7, "maxTokens": 500}
)

# 4. Parse response from Nova to inspect tool request
assistant_msg = initial_response["output"]["message"]
stop_reason = initial_response.get("stopReason")
print(f"Bedrock Stop Reason: {stop_reason}")

content_blocks = assistant_msg.get("content", [])
tool_use_block = next((block for block in content_blocks if "toolUse" in block), None)

if not tool_use_block:
    print("\n=== Nova Responded Directly Without Tool ===")
    print(content_blocks[0]["text"])
else:
    tool_use = tool_use_block["toolUse"]
    tool_name = tool_use["name"]
    tool_input = tool_use["input"]
    tool_use_id = tool_use["toolUseId"]
    
    print(f"\n=== Step 2: Amazon Nova Requested Tool Execution ===")
    print(f"Tool Name:   {tool_name}")
    print(f"Tool Use ID: {tool_use_id}")
    print(f"Tool Input:  {tool_input}")
    
    # 5. Invoke weather tool
    print("\n=== Step 3: Invoking Weather Tool ===")
    tool_output = invoke_weather_lambda(tool_input)
    print(f"Tool Output: {tool_output}")
    
    # Format natural language summary or structured JSON
    try:
        summary = f"The weather in {tool_output['location']} is currently {tool_output['temperature']} with {tool_output['condition']} (humidity: {tool_output.get('humidity', 'N/A')})."
    except Exception as e:
        summary = f"Weather data retrieved: {str(tool_output)}"
        
    # 6. Construct toolResult message
    tool_result_msg = {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content": [{"text": summary}],
                    "status": "success"
                }
            }
        ]
    }
    
    # 7. Pass toolResult message back to model with complete message history
    print("\n=== Step 4: Returning toolResult to Amazon Nova ===")
    final_response = bedrock.converse(
        modelId=MODEL_ID,
        system=system_prompt,
        messages=[initial_user_msg, assistant_msg, tool_result_msg],
        toolConfig=tool_config,
        inferenceConfig={"temperature": 0.7, "maxTokens": 500}
    )
    
    final_msg = final_response["output"]["message"]["content"][0]["text"]
    print("\n=== Final Agent Response ===")
    print(final_msg)
    print("\nFinal Turn Usage:", final_response.get("usage"))


=== Step 1: Sending Initial User Message to Amazon Nova ===
User: What's the weather like in portland?

Bedrock Stop Reason: end_turn

=== Nova Responded Directly Without Tool ===
<thinking> The user has provided a city name, "Portland", which exists in multiple geographic areas (e.g., Portland, Oregon, and Portland, Maine). I need to ask the user for clarification on which Portland they are referring to. </thinking>

Could you please specify which Portland you are referring to? There are multiple cities named Portland, such as Portland, Oregon, and Portland, Maine.


## 8. Advanced Patterns: Forced Tool Choice (`toolChoice`)

By default, `toolChoice: {"auto": {}}` allows the foundation model to decide whether to invoke a tool or reply directly with general knowledge.

In enterprise workflows, you often need to **enforce** that a tool is called:
- `{"any": {}}`: Forces the model to call *at least one* tool from `toolConfig.tools`.
- `{"tool": {"name": "getWeather"}}`: Forces the model to call the specific designated tool.

Let's test forced tool invocation with a question where the model might otherwise hesitate:

In [15]:
# Force the model to use getWeather
forced_tool_config = {
    "tools": [weather_tool],
    "toolChoice": {"tool": {"name": "getWeather"}}
}

query_tokyo = {"role": "user", "content": [{"text": "I am landing in Tokyo soon. Should I carry a coat?"}]}

print("Query with forced tool choice (toolChoice = {'tool': {'name': 'getWeather'}}):\n")
forced_resp = bedrock.converse(
    modelId=MODEL_ID,
    messages=[query_tokyo],
    toolConfig=forced_tool_config,
    inferenceConfig={"temperature": 0.0, "maxTokens": 300}
)

forced_assistant_msg = forced_resp["output"]["message"]
print("Stop Reason:", forced_resp.get("stopReason"))
for block in forced_assistant_msg.get("content", []):
    if "toolUse" in block:
        print("\nForced Tool Call Triggered:")
        print("  Name:", block["toolUse"]["name"])
        print("  Input Arguments:", block["toolUse"]["input"])


Query with forced tool choice (toolChoice = {'tool': {'name': 'getWeather'}}):

Stop Reason: tool_use

Forced Tool Call Triggered:
  Name: getWeather
  Input Arguments: {'lon': 139.6917, 'lat': 35.6895}


## 9. Clean Up & Resource Management

### Resource Lifecycle
- **On-Demand Converse API**: Stateless and pay-per-token. Incurs **$0.00 idle cost**.
- **AWS Lambda**: If you deployed the `weather-tool` Lambda function in your AWS console, you can delete it when finished from the AWS Lambda dashboard.
- **Kernel Memory**: Shut down your notebook kernel (`Kernel` -> `Shut Down Kernel`) when done to free local memory.

In [16]:
# Verify Lambda status in target region
try:
    func_info = lambda_client.get_function(FunctionName="weather-tool")
    print(f"Discovered AWS Lambda 'weather-tool' (ARN: {func_info['Configuration']['FunctionArn']})")
    print("To delete via Boto3: lambda_client.delete_function(FunctionName='weather-tool')")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceNotFoundException":
        print(f"No AWS Lambda function 'weather-tool' found in region {REGION}. Zero AWS resources to clean up!")
    else:
        print(f"Lambda check note: {e}")

print("\nClean up check completed successfully!")


No AWS Lambda function 'weather-tool' found in region us-east-2. Zero AWS resources to clean up!

Clean up check completed successfully!


## Summary

In this notebook, you explored:
- ✅ **The Core Building Blocks of the Converse API**: System prompts, structured message arrays, and universal inference parameters.
- ✅ **Amazon Model Integration**: Seamlessly invoked **Amazon Nova Pro (`us.amazon.nova-pro-v1:0`)** using standard System Inference Profiles (`us.amazon.nova-lite-v1:0`).
- ✅ **Context Window Management**: Token tracking via `usage` metrics and sliding window conversation pruning.
- ✅ **Weather Tool Specification**: Defined a full JSON Schema tool specification with coordinates (`lat`, `lon`).
- ✅ **Complete Tool Lifecycle**: Handled `stopReason='tool_use'`, dispatched execution, returned `toolResult`, and synthesized the final conversational answer.
- ✅ **Advanced Tool Control**: Explored `toolChoice: {'auto': {}}` vs `toolChoice: {'tool': ...}`.

### Complete Bedrock Exploration Series
- [001-Invoke-model.ipynb](001-Invoke-model.ipynb): Model discovery, synchronous invocations, prompt engineering.
- [001b-Converse-API.ipynb](001b-Converse-API.ipynb): Universal Converse API, streaming, and multimodal image/document analysis.
- [002-Streaming-response.ipynb](002-Streaming-response.ipynb): Streaming token lifecycle, TTFT metrics, and generator patterns.
- [003-Guardrails.ipynb](003-Guardrails.ipynb): Content moderation, PII redaction, topic filtering, and programmatic Guardrail APIs.
- [004-Knowledge-Base.ipynb](004-Knowledge-Base.ipynb): Retrieval-Augmented Generation (RAG) and Titan Text Embeddings V2.
- [005-Prompt-Management.ipynb](005-Prompt-Management.ipynb): Reusable prompt templates, versioning, and prompt optimization.
- [006-Tool-Calling.ipynb](006-Tool-Calling.ipynb): Math function tool calling with AWS Lambda.
- [007-Tool-Calling-Converse-API.ipynb](007-Tool-Calling-Converse-API.ipynb): In-depth Converse API building blocks, weather tool calling, and context management.
